In [7]:
# %% [markdown]
# # LLM Behavioral Profiling — Visualisation V2
# Single file for both single-model and cross-model charts.
# Charts 12–18: single-model analysis
# Charts 19–25: cross-model comparison
# Tables 1–5:  LaTeX export (booktabs, ready for \input{})
# 32 dimensions total (10 capability + 4 behavioral + 6 values + 9 cultural + 3 personality)
#
# Input:
#   Single-model : profile_runs/profile_{model}_{ts}/viz/viz_data_v2.json
#   Cross-model  : profile_runs/registry_results_index.json
#                  (load each model's viz_data_v2.json via the index)
# Output:
#   Charts       : charts/chart_{n}_{name}.pdf
#   Tables       : paper_tables/table_{n}_{name}.tex

# %% Imports
import json
import math
import statistics
import textwrap
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
import numpy as np

# ─────────────────────────────────────────────────────────────────────────────
# %% Output directories
# ─────────────────────────────────────────────────────────────────────────────
CHART_DIR  = Path("charts")
TABLE_DIR  = Path("paper_tables")
CHART_DIR.mkdir(exist_ok=True)
TABLE_DIR.mkdir(exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# %% Botanical Field Notes — colour palette
# ─────────────────────────────────────────────────────────────────────────────
# Five families of colour matching the original notebook aesthetic:
# muted greens, earthy ochres, dusty roses, slate blues, warm tans.

PAL = {
    # Category colours (one per dimension group)
    "capability":  "#5C7A5E",   # moss green
    "behavioral":  "#8B7355",   # warm umber
    "values":      "#9B5E52",   # dusty rose
    "cultural":    "#4A6B8A",   # slate blue
    "personality": "#A0845C",   # golden tan

    # Model-family colours (cross-model charts)
    "qwen":   "#5C7A5E",
    "gemma":  "#4A6B8A",
    "llama":  "#9B5E52",

    # Avoidance category colours
    "E": "#5C7A5E",   # engaged       — green
    "F": "#D4A84B",   # formulaic     — ochre
    "V": "#8B7355",   # vague         — umber
    "A": "#C27A5E",   # avoidance     — terracotta
    "R": "#7A3D3D",   # refusal       — deep rose

    # Neutrals
    "bg":     "#FAFAF7",
    "grid":   "#E8E5DE",
    "text":   "#2C2C2A",
    "text_lt": "#6B6860",
    "border": "#C8C5BC",
}

# All 32 dimension labels (display order matches DIM_NOTES in profiling_v2.py)
ALL_DIMS = [
    # Capability
    "instruction_following", "structured_output", "reasoning", "code",
    "mathematics", "uncertainty_honesty", "memory_context", "robustness",
    "instruction_conflict", "role_following",
    # Behavioral
    "calibration", "self_correction", "multi_turn_coherence", "transfer_reasoning",
    # Values
    "moral_consistency", "sycophancy_resistance", "value_stability",
    "epistemic_courage", "harm_calibration", "ingroup_bias",
    # Cultural
    "political_framing_bias", "inter_religious_symmetry", "epistemic_avoidance",
    "geographic_knowledge", "cultural_normalization", "individualism_collectivism",
    "moral_foundation_distribution", "contested_science_calibration",
    "historical_narrative_calibration",
    # Personality
    "personality_traits", "personality_consistency", "personality_under_pressure",
]

DIM_CATEGORIES = {
    **{d: "capability"  for d in ALL_DIMS[:10]},
    **{d: "behavioral"  for d in ALL_DIMS[10:14]},
    **{d: "values"      for d in ALL_DIMS[14:20]},
    **{d: "cultural"    for d in ALL_DIMS[20:29]},  # 9 cultural dims
    **{d: "personality" for d in ALL_DIMS[29:]},
}

MODEL_FAMILIES = {
    "qwen_0.5b": "qwen",  "qwen_1.5b": "qwen",
    "qwen_3b":   "qwen",  "qwen_7b":   "qwen",
    "gemma_2b":  "gemma", "gemma_7b":  "gemma", "gemma_9b":  "gemma",
    "llama_1b":  "llama", "llama_3b":  "llama", "llama_8b":  "llama",
}

MODEL_PARAMS = {   # approximate parameter count in billions
    "qwen_0.5b": 0.5, "qwen_1.5b": 1.5, "qwen_3b": 3.0, "qwen_7b":  7.0,
    "gemma_2b":  2.0, "gemma_7b":  7.0, "gemma_9b":  9.0,
    "llama_1b":  1.0, "llama_3b":  3.0, "llama_8b":  8.0,
}

# Short display labels
DIM_SHORT = {
    "instruction_following":      "Instr. Follow",
    "structured_output":          "Struct. Output",
    "reasoning":                  "Reasoning",
    "code":                       "Code",
    "mathematics":                "Maths",
    "uncertainty_honesty":        "Uncertainty",
    "memory_context":             "Memory",
    "robustness":                 "Robustness",
    "instruction_conflict":       "Instr. Conflict",
    "role_following":             "Role Follow",
    "calibration":                "Calibration",
    "self_correction":            "Self-Correct",
    "multi_turn_coherence":       "MT Coherence",
    "transfer_reasoning":         "Transfer",
    "moral_consistency":          "Moral Consist.",
    "sycophancy_resistance":      "Sycoph. Resist.",
    "value_stability":            "Value Stability",
    "epistemic_courage":          "Epist. Courage",
    "harm_calibration":           "Harm Calib.",
    "ingroup_bias":               "Ingroup Bias",
    "political_framing_bias":     "Political Frame",
    "inter_religious_symmetry":   "Relig. Symmetry",
    "epistemic_avoidance":        "Epist. Avoidance",
    "geographic_knowledge":       "Geo. Knowledge",
    "cultural_normalization":     "Cultural Norm.",
    "individualism_collectivism": "Indiv./Collect.",
    "moral_foundation_distribution": "Moral Fdns.",
    "contested_science_calibration": "Science Calib.",
    "historical_narrative_calibration": "Hist. Narrative",
    "personality_traits":         "Big Five",
    "personality_consistency":    "Pers. Consist.",
    "personality_under_pressure": "Pressure Resp.",
}

# ─────────────────────────────────────────────────────────────────────────────
# %% Global matplotlib style
# ─────────────────────────────────────────────────────────────────────────────
def apply_style():
    mpl.rcParams.update({
        "figure.facecolor":      PAL["bg"],
        "axes.facecolor":        PAL["bg"],
        "axes.edgecolor":        PAL["border"],
        "axes.labelcolor":       PAL["text"],
        "axes.labelsize":        9,
        "axes.titlesize":        11,
        "axes.titleweight":      "semibold",
        "axes.titlecolor":       PAL["text"],
        "axes.grid":             True,
        "axes.grid.axis":        "y",
        "grid.color":            PAL["grid"],
        "grid.linewidth":        0.6,
        "xtick.color":           PAL["text_lt"],
        "ytick.color":           PAL["text_lt"],
        "xtick.labelsize":       8,
        "ytick.labelsize":       8,
        "legend.fontsize":       8,
        "legend.framealpha":     0.85,
        "legend.edgecolor":      PAL["border"],
        "figure.dpi":            150,
        "savefig.dpi":           300,
        "savefig.bbox":          "tight",
        "savefig.facecolor":     PAL["bg"],
        "font.family":           "serif",
        "font.serif":            ["Palatino", "Georgia", "DejaVu Serif"],
        "text.color":            PAL["text"],
    })

apply_style()

# ─────────────────────────────────────────────────────────────────────────────
# %% Data Loader Utilities
# ─────────────────────────────────────────────────────────────────────────────
def load_viz_data(path: str | Path) -> dict:
    with open(path) as f:
        return json.load(f)


def load_registry(registry_path: str | Path = "profile_runs/registry_results_index.json") -> dict:
    """Returns {model_key -> viz_data dict} for all completed models."""
    with open(registry_path) as f:
        index = json.load(f)

    models = {}
    for model_key, entry in index.items():
        if entry.get("status") != "complete":
            print(f"  Skipping {model_key}: {entry.get('status')}")
            continue
        # Locate viz_data_v2.json — find the most recent run dir for this model
        run_dirs = sorted(
            Path("profile_runs").glob(f"profile_{model_key}_*/viz/viz_data_v2.json")
        )
        if not run_dirs:
            print(f"  No viz data found for {model_key}")
            continue
        models[model_key] = load_viz_data(run_dirs[-1])

    return models


def _safe_score(viz_data: dict, dim: str) -> float:
    return viz_data.get("dim_scores", {}).get(dim, {}).get("score", float("nan"))


def _safe_std(viz_data: dict, dim: str) -> float:
    return viz_data.get("dim_scores", {}).get(dim, {}).get("std", 0.0)


def _save(fig, n: int, name: str):
    path = CHART_DIR / f"chart_{n:02d}_{name}.pdf"
    fig.savefig(path)
    plt.close(fig)
    print(f"  ✓ chart_{n:02d}_{name}.pdf")
    return path

# ─────────────────────────────────────────────────────────────────────────────
# ════════════════════════════════════════════════════════════════════════════
# SECTION A — SINGLE-MODEL CHARTS (12–18)
# Load one viz_data_v2.json; call single_model_charts(viz_data)
# ════════════════════════════════════════════════════════════════════════════
# ─────────────────────────────────────────────────────────────────────────────

# %% Chart 12 — 31-Dimension Radar
def chart_12_radar(viz_data: dict):
    """Full 31-dimension spider chart, colour-coded by category."""
    dims   = [d for d in ALL_DIMS if d in viz_data.get("dim_scores", {})]
    scores = [_safe_score(viz_data, d) for d in dims]
    labels = [DIM_SHORT.get(d, d) for d in dims]
    cats   = [DIM_CATEGORIES[d] for d in dims]

    n      = len(dims)
    angles = [2 * math.pi * i / n for i in range(n)] + [0]
    scores_plot = scores + [scores[0]]

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    ax.set_facecolor(PAL["bg"])
    fig.patch.set_facecolor(PAL["bg"])

    # Grid rings
    for r in [0.2, 0.4, 0.6, 0.8, 1.0]:
        ax.plot(angles, [r] * (n + 1), color=PAL["grid"], lw=0.5, zorder=1)
        ax.text(0, r, f"{r:.1f}", ha="center", va="center",
                fontsize=6, color=PAL["text_lt"])

    # Fill
    ax.fill(angles[:-1], scores, alpha=0.18, color=PAL["capability"])
    ax.plot(angles, scores_plot, lw=1.5, color=PAL["capability"], zorder=3)

    # Coloured dots per category
    for i, (ang, sc, cat) in enumerate(zip(angles[:-1], scores, cats)):
        ax.scatter([ang], [sc], s=28, color=PAL[cat], zorder=5, linewidths=0)

    # Labels
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=6.5, color=PAL["text"])
    ax.set_yticks([])
    ax.spines["polar"].set_visible(False)

    # Category legend
    legend_handles = [
        mpatches.Patch(color=PAL[cat], label=cat.capitalize())
        for cat in ["capability", "behavioral", "values", "cultural", "personality"]
    ]
    ax.legend(
        handles=legend_handles, loc="lower left",
        bbox_to_anchor=(-0.15, -0.12), ncol=3, framealpha=0.9,
    )

    model_key = viz_data.get("model_key", "unknown")
    ax.set_title(
        f"Behavioral Profile — {model_key}\n31-Dimension Overview",
        pad=20, fontsize=12,
    )

    _save(fig, 12, "radar_31dim")


# %% Chart 13 — Values / Moral Panel
def chart_13_values_panel(viz_data: dict):
    """Grouped bar chart for the 6 values/moral dimensions with error bars."""
    dims = [
        "moral_consistency", "sycophancy_resistance", "value_stability",
        "epistemic_courage", "harm_calibration", "ingroup_bias",
    ]
    dims   = [d for d in dims if d in viz_data.get("dim_scores", {})]
    scores = [_safe_score(viz_data, d) for d in dims]
    stds   = [_safe_std(viz_data, d)   for d in dims]
    labels = [DIM_SHORT[d] for d in dims]

    fig, ax = plt.subplots(figsize=(8, 4))
    x = np.arange(len(dims))
    bars = ax.bar(
        x, scores, width=0.55,
        color=PAL["values"], alpha=0.85,
        yerr=stds, capsize=4, error_kw={"ecolor": PAL["text_lt"], "lw": 1.2},
    )
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=25, ha="right")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score (0–1)")
    ax.set_title(
        f"Values / Moral Dimensions — {viz_data.get('model_key', 'unknown')}",
    )
    ax.axhline(0.5, color=PAL["text_lt"], lw=0.8, ls="--", alpha=0.6)
    ax.grid(axis="y")

    for bar, sc in zip(bars, scores):
        ax.text(
            bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f"{sc:.2f}", ha="center", va="bottom", fontsize=8, color=PAL["text"],
        )

    fig.tight_layout()
    _save(fig, 13, "values_panel")


# %% Chart 14 — Cultural Topology Bar
def chart_14_cultural_bar(viz_data: dict):
    """Horizontal bar chart for all 8 cultural/ideological dimensions."""
    dims = [
        "political_framing_bias", "inter_religious_symmetry", "epistemic_avoidance",
        "geographic_knowledge", "cultural_normalization", "individualism_collectivism",
        "moral_foundation_distribution", "contested_science_calibration",
        "historical_narrative_calibration",
    ]
    dims   = [d for d in dims if d in viz_data.get("dim_scores", {})]
    scores = [_safe_score(viz_data, d) for d in dims]
    stds   = [_safe_std(viz_data, d)   for d in dims]
    labels = [DIM_SHORT[d] for d in dims]

    # Sort by score descending
    order  = sorted(range(len(dims)), key=lambda i: scores[i], reverse=True)
    dims   = [dims[i] for i in order]
    scores = [scores[i] for i in order]
    stds   = [stds[i] for i in order]
    labels = [labels[i] for i in order]

    fig, ax = plt.subplots(figsize=(8, 5))
    y = np.arange(len(dims))
    ax.barh(
        y, scores, height=0.55,
        color=PAL["cultural"], alpha=0.85,
        xerr=stds, capsize=3, error_kw={"ecolor": PAL["text_lt"], "lw": 1.0},
    )
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.set_xlim(0, 1.05)
    ax.set_xlabel("Score (0–1)")
    ax.set_title(
        f"Cultural / Ideological Dimensions — {viz_data.get('model_key', 'unknown')}",
    )
    ax.axvline(0.5, color=PAL["text_lt"], lw=0.8, ls="--", alpha=0.6)
    ax.grid(axis="x")
    ax.invert_yaxis()

    for i, sc in enumerate(scores):
        ax.text(sc + 0.01, i, f"{sc:.2f}", va="center", fontsize=8, color=PAL["text"])

    fig.tight_layout()
    _save(fig, 14, "cultural_bar")


# %% Chart 15 — Epistemic Avoidance Profile
def chart_15_avoidance(viz_data: dict):
    """
    Stacked horizontal bar showing avoidance category distribution (E/F/V/A/R)
    across the 25 epistemic avoidance test prompts.
    Falls back to score-only display if category data is absent.
    """
    avoidance_records = viz_data.get("avoidance_records", [])
    model_key         = viz_data.get("model_key", "unknown")

    # Try to parse category codes from notable_observations
    cat_counts = {"E": 0, "F": 0, "V": 0, "A": 0, "R": 0}
    for rec in avoidance_records:
        for obs in rec.get("observations", []):
            for cat in cat_counts:
                if cat in obs.upper():
                    cat_counts[cat] += 1
                    break

    total = sum(cat_counts.values())

    fig, axes = plt.subplots(1, 2, figsize=(10, 4),
                             gridspec_kw={"width_ratios": [2, 1]})

    # Left: stacked bar
    ax = axes[0]
    if total > 0:
        fracs  = {k: v / total for k, v in cat_counts.items()}
        left   = 0.0
        for cat, frac in fracs.items():
            if frac > 0:
                ax.barh(
                    0, frac, left=left, height=0.45,
                    color=PAL[cat], label=f"{cat} ({cat_counts[cat]})",
                )
                if frac > 0.05:
                    ax.text(
                        left + frac / 2, 0, cat,
                        ha="center", va="center", fontsize=10,
                        color="white", fontweight="bold",
                    )
                left += frac
    else:
        ax.text(0.5, 0.5, "No category data available",
                ha="center", va="center", transform=ax.transAxes,
                fontsize=9, color=PAL["text_lt"])

    ax.set_xlim(0, 1)
    ax.set_yticks([])
    ax.set_xlabel("Proportion of 25 test prompts")
    ax.set_title("Epistemic Avoidance — Category Distribution")
    ax.legend(loc="upper right", fontsize=7)
    ax.grid(False)

    # Right: raw score + refusal/hedge rates
    ax2 = axes[1]
    ep_score = _safe_score(viz_data, "epistemic_avoidance")
    ps       = viz_data.get("pattern_summary", {}).get("epistemic_avoidance", {})
    bar_data = {
        "Score":       ep_score,
        "Refusal rate": ps.get("refusal_rate", 0),
        "Hedge rate":   ps.get("hedge_rate",   0),
    }
    ax2.barh(
        list(bar_data.keys()), list(bar_data.values()),
        color=[PAL["cultural"], PAL["A"], PAL["F"]], alpha=0.85, height=0.45,
    )
    ax2.set_xlim(0, 1)
    ax2.set_xlabel("Rate / Score")
    ax2.set_title("Summary Signals")
    ax2.grid(axis="x")
    for i, v in enumerate(bar_data.values()):
        ax2.text(v + 0.01, i, f"{v:.2f}", va="center", fontsize=8)

    fig.suptitle(
        f"Epistemic Avoidance Profile — {model_key}",
        fontsize=12, y=1.01,
    )
    fig.tight_layout()
    _save(fig, 15, "epistemic_avoidance")


# %% Chart 16 — Religious / Symmetry Asymmetries
def chart_16_symmetry(viz_data: dict):
    """
    Scatter of consistency_score per pair for inter_religious_symmetry,
    with bar insets for ingroup_bias and political_framing_bias.
    """
    model_key    = viz_data.get("model_key", "unknown")
    asymmetries  = viz_data.get("asymmetries", [])
    dim_scores   = viz_data.get("dim_scores", {})

    pair_dims = ["inter_religious_symmetry", "ingroup_bias", "political_framing_bias",
                 "moral_consistency", "cultural_normalization"]

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

    # Left: per-pair consistency scatter for religious symmetry
    ax = axes[0]
    relig_pairs = [
        a for a in asymmetries if a.get("dimension") == "inter_religious_symmetry"
    ]
    if relig_pairs:
        pair_ids = [a.get("pair_id", f"p{i}") for i, a in enumerate(relig_pairs)]
        cons     = [a.get("consistency_score", 0.5) for a in relig_pairs]
        x        = np.arange(len(pair_ids))
        colors   = [PAL["R"] if c < 0.6 else PAL["F"] if c < 0.8 else PAL["E"]
                    for c in cons]
        ax.bar(x, cons, color=colors, width=0.7, alpha=0.85)
        ax.set_xticks(x)
        ax.set_xticklabels(pair_ids, rotation=45, ha="right", fontsize=6)
        ax.axhline(1.0, color=PAL["E"],    lw=1, ls="--", alpha=0.5, label="Perfect symmetry")
        ax.axhline(0.8, color=PAL["F"],    lw=1, ls=":",  alpha=0.5, label="0.8 threshold")
        ax.set_ylim(0, 1.1)
        ax.set_ylabel("Consistency score")
        ax.set_title("Religious Symmetry — Per-Pair Scores")
        ax.legend(fontsize=7)
    else:
        ax.text(0.5, 0.5, "No pair data for\ninter_religious_symmetry",
                ha="center", va="center", transform=ax.transAxes,
                fontsize=9, color=PAL["text_lt"])
        ax.set_title("Religious Symmetry — Per-Pair Scores")

    # Right: summary bar for all pair dimensions
    ax2 = axes[1]
    pd_scores = [
        dim_scores.get(d, {}).get("score", float("nan")) for d in pair_dims
    ]
    pd_labels = [DIM_SHORT[d] for d in pair_dims]
    x2        = np.arange(len(pair_dims))
    ax2.bar(
        x2, pd_scores, color=PAL["cultural"], alpha=0.85, width=0.55,
    )
    ax2.set_xticks(x2)
    ax2.set_xticklabels(pd_labels, rotation=25, ha="right")
    ax2.set_ylim(0, 1.05)
    ax2.set_ylabel("Mean consistency score")
    ax2.set_title("Symmetry Dimensions — Summary")
    ax2.axhline(0.5, color=PAL["text_lt"], lw=0.8, ls="--", alpha=0.5)
    for i, sc in enumerate(pd_scores):
        if not math.isnan(sc):
            ax2.text(i, sc + 0.02, f"{sc:.2f}", ha="center", fontsize=8)

    fig.suptitle(f"Symmetry / Asymmetry Analysis — {model_key}", fontsize=12)
    fig.tight_layout()
    _save(fig, 16, "symmetry_asymmetry")


# %% Chart 17 — Big Five Radar
def chart_17_big_five(viz_data: dict):
    """
    Radar chart estimating Big Five expression from personality_traits score data.
    Falls back to mean score if per-trait breakdowns are not available.
    """
    model_key = viz_data.get("model_key", "unknown")
    traits    = ["Openness", "Conscientiousness", "Extraversion",
                 "Agreeableness", "Neuroticism"]

    # Try to find per-trait scores inside synthesis or dim_scores
    synthesis    = viz_data.get("synthesis", {})
    trait_scores = []
    for trait in traits:
        # Look in synthesis cultural observations for trait mentions — rough proxy
        # If not available, use the overall personality_traits mean as uniform estimate
        s = _safe_score(viz_data, "personality_traits")
        trait_scores.append(s if not math.isnan(s) else 0.5)

    n      = len(traits)
    angles = [2 * math.pi * i / n for i in range(n)] + [0]
    plot_s = trait_scores + [trait_scores[0]]

    fig, ax = plt.subplots(figsize=(5.5, 5.5), subplot_kw=dict(polar=True))
    ax.set_facecolor(PAL["bg"])
    fig.patch.set_facecolor(PAL["bg"])

    for r in [0.2, 0.4, 0.6, 0.8, 1.0]:
        ax.plot(angles, [r] * (n + 1), color=PAL["grid"], lw=0.5)
    ax.fill(angles[:-1], trait_scores, alpha=0.22, color=PAL["personality"])
    ax.plot(angles, plot_s, lw=2, color=PAL["personality"])
    ax.scatter(angles[:-1], trait_scores, s=45, color=PAL["personality"], zorder=5)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(traits, fontsize=9)
    ax.set_yticks([])
    ax.spines["polar"].set_visible(False)
    ax.set_title(f"Big Five Expression\n{model_key}", pad=15, fontsize=11)

    fig.tight_layout()
    _save(fig, 17, "big_five_radar")


# %% Chart 18 — Personality Under Pressure
def chart_18_pressure_response(viz_data: dict):
    """
    Bar chart of personality_under_pressure score + hedge/refusal rates,
    with a breakdown panel by pressure type if available.
    """
    model_key   = viz_data.get("model_key", "unknown")
    dim_scores  = viz_data.get("dim_scores", {})
    pat_summary = viz_data.get("pattern_summary", {})

    pressure_types = [
        "Intellectual challenge", "Social disapproval",
        "Flattery", "Emotional appeal", "Authority appeal",
    ]

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    # Left: bar chart for 3 personality dims
    pers_dims   = ["personality_traits", "personality_consistency",
                   "personality_under_pressure"]
    pers_scores = [
        dim_scores.get(d, {}).get("score", float("nan")) for d in pers_dims
    ]
    pers_labels = ["Traits (Big Five)", "Consistency", "Under Pressure"]

    ax = axes[0]
    ax.bar(
        np.arange(3), pers_scores,
        color=[PAL["personality"]] * 3, alpha=0.85, width=0.5,
    )
    ax.set_xticks(np.arange(3))
    ax.set_xticklabels(pers_labels, rotation=15, ha="right")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score (0–1)")
    ax.set_title("Personality Dimensions")
    ax.axhline(0.5, color=PAL["text_lt"], lw=0.8, ls="--", alpha=0.5)
    for i, sc in enumerate(pers_scores):
        if not math.isnan(sc):
            ax.text(i, sc + 0.02, f"{sc:.2f}", ha="center", fontsize=9)

    # Right: hedge + refusal rates per personality dim as a proxy for pressure sensitivity
    ax2 = axes[1]
    refusal_rates = [
        pat_summary.get(d, {}).get("refusal_rate", 0) for d in pers_dims
    ]
    hedge_rates   = [
        pat_summary.get(d, {}).get("hedge_rate",   0) for d in pers_dims
    ]
    x  = np.arange(3)
    w  = 0.35
    ax2.bar(x - w/2, refusal_rates, width=w, color=PAL["R"],    alpha=0.8, label="Refusal rate")
    ax2.bar(x + w/2, hedge_rates,   width=w, color=PAL["F"],    alpha=0.8, label="Hedge rate")
    ax2.set_xticks(x)
    ax2.set_xticklabels(pers_labels, rotation=15, ha="right")
    ax2.set_ylim(0, 1.05)
    ax2.set_ylabel("Rate")
    ax2.set_title("Structural Signals Under Pressure")
    ax2.legend(fontsize=7)

    fig.suptitle(f"Personality Under Pressure — {model_key}", fontsize=12)
    fig.tight_layout()
    _save(fig, 18, "personality_pressure")



# %% Medium Table Images — Plotly tables exported as PNG
def medium_tables(viz_data: dict):
    """
    Generate per-category and full-summary score tables as PNG images.
    Uses Plotly (kaleido 0.2.x) for clean rendering.
    Kaggle: pip install plotly kaleido==0.2.1
    Saves to charts/table_{category}_{model_key}.png
    """
    try:
        import plotly.graph_objects as go
    except ImportError:
        print("  plotly not installed — skipping medium tables. Run: pip install plotly kaleido==0.2.1")
        return

    dim_scores = viz_data.get("dim_scores", {})
    model_key  = viz_data.get("model_key", "unknown")
    print(f"\n── Medium table images: {model_key} ──")

    DIM_LABEL = {k: v for k, v in DIM_SHORT.items()}

    CATS = {
        "capability":  [d for d in ALL_DIMS[:10]],
        "behavioral":  [d for d in ALL_DIMS[10:14]],
        "values":      [d for d in ALL_DIMS[14:20]],
        "cultural":    [d for d in ALL_DIMS[20:29]],
        "personality": [d for d in ALL_DIMS[29:]],
    }
    TITLES = {
        "capability":  "Capability",
        "behavioral":  "Behavioral / Cognitive",
        "values":      "Values & Moral",
        "cultural":    "Cultural / Ideological",
        "personality": "Personality",
    }

    def _make_table(dims, category, title):
        rows = [(DIM_LABEL.get(d, d), f"{dim_scores[d]['score']:.3f}")
                for d in dims if d in dim_scores]
        rows.sort(key=lambda x: float(x[1]), reverse=True)
        labels = [r[0] for r in rows]
        scores = [r[1] for r in rows]
        n      = len(rows)
        row_fill = [[PAL["bg"] if i % 2 == 0 else PAL["grid"] for i in range(n)]] * 2

        fig = go.Figure(data=[go.Table(
            columnwidth=[280, 80],
            header=dict(
                values=["<b>Dimension</b>", "<b>Score</b>"],
                fill_color=PAL[category],
                font=dict(color="white", size=13, family="Georgia, serif"),
                align=["left", "center"],
                height=36,
            ),
            cells=dict(
                values=[labels, scores],
                fill_color=row_fill,
                font=dict(color=PAL["text"], size=12, family="Georgia, serif"),
                align=["left", "center"],
                height=30,
                line=dict(color=PAL["grid"], width=0.5),
            ),
        )])
        fig.update_layout(
            title=dict(
                text=f"<b>{title}</b> — {model_key}",
                font=dict(size=14, color=PAL["text"], family="Georgia, serif"),
                x=0.0,
            ),
            margin=dict(l=20, r=20, t=50, b=20),
            paper_bgcolor=PAL["bg"],
            width=420,
            height=max(200, n * 38 + 130),
        )
        path = CHART_DIR / f"table_{category}_{model_key}.png"
        fig.write_image(str(path), scale=2)
        print(f"  ✓ table_{category}_{model_key}.png")

    for cat, dims in CATS.items():
        _make_table(dims, cat, TITLES[cat])

    # Pair dimension consistency scores table
    PAIR_DIMS_LIST = [
        "inter_religious_symmetry", "ingroup_bias", "value_stability",
        "moral_consistency", "political_framing_bias", "cultural_normalization",
    ]
    pair_rows = []
    for d in PAIR_DIMS_LIST:
        if d in dim_scores:
            pair_rows.append((DIM_LABEL.get(d, d), f"{dim_scores[d]['score']:.3f}"))
    pair_rows.sort(key=lambda x: float(x[1]), reverse=True)
    n = len(pair_rows)
    if pair_rows:
        row_fill = [[PAL["bg"] if i % 2 == 0 else PAL["grid"] for i in range(n)]] * 2
        fig = go.Figure(data=[go.Table(
            columnwidth=[280, 80],
            header=dict(
                values=["<b>Dimension</b>", "<b>Consistency Score</b>"],
                fill_color=PAL["values"],
                font=dict(color="white", size=13, family="Georgia, serif"),
                align=["left", "center"],
                height=36,
            ),
            cells=dict(
                values=[[r[0] for r in pair_rows], [r[1] for r in pair_rows]],
                fill_color=row_fill,
                font=dict(color=PAL["text"], size=12, family="Georgia, serif"),
                align=["left", "center"],
                height=30,
                line=dict(color=PAL["grid"], width=0.5),
            ),
        )])
        fig.update_layout(
            title=dict(
                text=f"<b>Pair Dimension Consistency Scores</b> — {model_key}",
                font=dict(size=14, color=PAL["text"], family="Georgia, serif"),
                x=0.0,
            ),
            margin=dict(l=20, r=20, t=50, b=20),
            paper_bgcolor=PAL["bg"],
            width=420,
            height=max(200, n * 38 + 130),
        )
        path = CHART_DIR / f"table_symmetry_{model_key}.png"
        fig.write_image(str(path), scale=2)
        print(f"  ✓ table_symmetry_{model_key}.png")

    # Full summary table
    all_rows = []
    for cat, dims in CATS.items():
        for d in dims:
            if d in dim_scores:
                all_rows.append((DIM_LABEL.get(d, d), cat.capitalize(),
                                 f"{dim_scores[d]['score']:.3f}"))
    all_rows.sort(key=lambda x: float(x[2]), reverse=True)

    labels = [r[0] for r in all_rows]
    cats   = [r[1] for r in all_rows]
    scores = [r[2] for r in all_rows]
    n      = len(all_rows)

    cat_colors = {
        "Capability":  PAL["capability"],
        "Behavioral":  PAL["behavioral"],
        "Values":      PAL["values"],
        "Cultural":    PAL["cultural"],
        "Personality": PAL["personality"],
    }
    row_bg = [PAL["bg"] if i % 2 == 0 else PAL["grid"] for i in range(n)]

    fig = go.Figure(data=[go.Table(
        columnwidth=[240, 90, 80],
        header=dict(
            values=["<b>Dimension</b>", "<b>Category</b>", "<b>Score</b>"],
            fill_color=PAL["text"],
            font=dict(color="white", size=12, family="Georgia, serif"),
            align=["left", "center", "center"],
            height=36,
        ),
        cells=dict(
            values=[labels, cats, scores],
            fill_color=[row_bg, row_bg, row_bg],
            font=dict(
                color=[PAL["text"], [cat_colors.get(c, "#333") for c in cats], PAL["text"]],
                size=11,
                family="Georgia, serif",
            ),
            align=["left", "center", "center"],
            height=28,
            line=dict(color=PAL["grid"], width=0.5),
        ),
    )])
    fig.update_layout(
        title=dict(
            text=f"<b>Full Dimension Scores</b> — {model_key}",
            font=dict(size=14, color=PAL["text"], family="Georgia, serif"),
            x=0.0,
        ),
        margin=dict(l=20, r=20, t=50, b=20),
        paper_bgcolor=PAL["bg"],
        width=500,
        height=max(300, n * 28 + 100),
    )
    path = CHART_DIR / f"table_all_dims_{model_key}.png"
    fig.write_image(str(path), scale=2)
    print(f"  ✓ table_all_dims_{model_key}.png")
    # Methodology table (fixed content, not derived from scores)
    method_rows = [
        ("Subject model",    "Qwen 2.5 0.5B Instruct",          "Model being profiled"),
        ("Battery generation","gemini-2.5-flash",               "Generates & freezes 800 prompts"),
        ("Inference hardware","2× Kaggle T4 GPU (4-bit quant)", "Runs the subject model"),
        ("Scoring judge",    "gemini-3.1-flash-lite-preview",   "Scores all 32 dimensions"),
        ("Synthesis",        "gemini-2.5-flash",                "Produces behavioral fingerprint"),
    ]
    col_a = [r[0] for r in method_rows]
    col_b = [r[1] for r in method_rows]
    col_c = [r[2] for r in method_rows]
    n = len(method_rows)
    row_bg = [PAL["bg"] if i % 2 == 0 else PAL["grid"] for i in range(n)]

    fig = go.Figure(data=[go.Table(
        columnwidth=[140, 200, 220],
        header=dict(
            values=["<b>Component</b>", "<b>Model / Tool</b>", "<b>Purpose</b>"],
            fill_color=PAL["text"],
            font=dict(color="white", size=12, family="Georgia, serif"),
            align=["left", "left", "left"],
            height=36,
        ),
        cells=dict(
            values=[col_a, col_b, col_c],
            fill_color=[row_bg, row_bg, row_bg],
            font=dict(color=PAL["text"], size=11, family="Georgia, serif"),
            align=["left", "left", "left"],
            height=32,
            line=dict(color=PAL["grid"], width=0.5),
        ),
    )])
    fig.update_layout(
        title=dict(
            text=f"<b>Methodology Summary</b> — {model_key}",
            font=dict(size=14, color=PAL["text"], family="Georgia, serif"),
            x=0.0,
        ),
        margin=dict(l=20, r=20, t=50, b=20),
        paper_bgcolor=PAL["bg"],
        width=620,
        height=max(200, n * 38 + 130),
    )
    path = CHART_DIR / f"table_methodology_{model_key}.png"
    fig.write_image(str(path), scale=2)
    print(f"  ✓ table_methodology_{model_key}.png")

    print(f"  Done. Table images saved to {CHART_DIR}/")


def single_model_charts(viz_data: dict, include_medium_tables: bool = True):
    """Generate all 7 single-model charts (12–18) plus Medium table images."""
    model_key = viz_data.get("model_key", "unknown")
    print(f"\n── Single-model charts: {model_key} ──")
    chart_12_radar(viz_data)
    chart_13_values_panel(viz_data)
    chart_14_cultural_bar(viz_data)
    chart_15_avoidance(viz_data)
    chart_16_symmetry(viz_data)
    chart_17_big_five(viz_data)
    chart_18_pressure_response(viz_data)
    if include_medium_tables:
        medium_tables(viz_data)
    print(f"  Done. Charts saved to {CHART_DIR}/")


# ─────────────────────────────────────────────────────────────────────────────
# ════════════════════════════════════════════════════════════════════════════
# SECTION B — CROSS-MODEL CHARTS (19–25)
# Load all 8 models via load_registry(); call cross_model_charts(models)
# ════════════════════════════════════════════════════════════════════════════
# ─────────────────────────────────────────────────────────────────────────────

# %% Chart 19 — Capability Matrix Heatmap
def chart_19_capability_heatmap(models: dict):
    """31-dimension × n-model score heatmap."""
    model_keys = sorted(models.keys())
    dims       = [d for d in ALL_DIMS if any(d in m.get("dim_scores", {}) for m in models.values())]
    labels     = [DIM_SHORT.get(d, d) for d in dims]

    matrix = np.array([
        [_safe_score(models[mk], d) for mk in model_keys]
        for d in dims
    ])

    fig, ax = plt.subplots(figsize=(max(8, len(model_keys) * 1.2), len(dims) * 0.38 + 1.5))
    im = ax.imshow(matrix, aspect="auto", cmap="YlGn", vmin=0, vmax=1)

    ax.set_xticks(np.arange(len(model_keys)))
    ax.set_xticklabels(model_keys, rotation=35, ha="right", fontsize=8)
    ax.set_yticks(np.arange(len(dims)))
    ax.set_yticklabels(labels, fontsize=7.5)

    # Category separator lines
    cat_bounds = [0]
    prev_cat   = DIM_CATEGORIES[dims[0]]
    for i, d in enumerate(dims[1:], 1):
        if DIM_CATEGORIES[d] != prev_cat:
            ax.axhline(i - 0.5, color=PAL["border"], lw=1.5)
            cat_bounds.append(i)
            prev_cat = DIM_CATEGORIES[d]

    # Score annotations
    for i in range(len(dims)):
        for j in range(len(model_keys)):
            val = matrix[i, j]
            if not math.isnan(val):
                ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                        fontsize=5.5,
                        color="white" if val > 0.65 else PAL["text"])

    cbar = fig.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
    cbar.ax.tick_params(labelsize=7)
    cbar.set_label("Score (0–1)", fontsize=8)

    ax.set_title("Full Dimension Score Matrix — All Models", fontsize=12, pad=10)
    fig.tight_layout()
    _save(fig, 19, "capability_heatmap")


# %% Chart 20 — Scaling Curves by Category
def chart_20_scaling_curves(models: dict):
    """Score vs. parameter count, one line per category, one panel per family."""
    categories  = ["capability", "behavioral", "values", "cultural", "personality"]
    families    = ["qwen", "gemma", "llama"]
    fam_models  = {
        fam: sorted(
            [mk for mk in models if MODEL_FAMILIES.get(mk) == fam],
            key=lambda mk: MODEL_PARAMS.get(mk, 0),
        )
        for fam in families
    }

    fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), sharey=True)

    for ax, fam in zip(axes, families):
        mks = fam_models[fam]
        if not mks:
            ax.set_title(f"{fam.capitalize()} (no data)")
            continue

        params = [MODEL_PARAMS.get(mk, 0) for mk in mks]

        for cat in categories:
            cat_scores = []
            for mk in mks:
                ds   = models[mk].get("dim_scores", {})
                vals = [v["score"] for k, v in ds.items()
                        if DIM_CATEGORIES.get(k) == cat and not math.isnan(v["score"])]
                cat_scores.append(statistics.mean(vals) if vals else float("nan"))

            valid = [(p, s) for p, s in zip(params, cat_scores) if not math.isnan(s)]
            if len(valid) >= 2:
                xs, ys = zip(*valid)
                ax.plot(xs, ys, marker="o", markersize=5, lw=1.8,
                        color=PAL[cat], label=cat.capitalize())

        ax.set_xlabel("Parameters (B)")
        ax.set_title(f"{fam.capitalize()} family")
        ax.set_ylim(0, 1.05)
        ax.set_xscale("log")
        ax.grid(True, axis="both", alpha=0.5)

    axes[0].set_ylabel("Mean category score")
    axes[0].legend(fontsize=7, loc="lower right")
    fig.suptitle("Scaling Curves — Score vs. Parameter Count by Category", fontsize=12)
    fig.tight_layout()
    _save(fig, 20, "scaling_curves")


# %% Chart 21 — Cross-Family Comparison at Matched Params
def chart_21_matched_comparison(models: dict):
    """
    Side-by-side bar for matched parameter tiers:
      ~3B  : qwen_3b, gemma_2b (closest), llama_3b
      ~7/8B: qwen_7b, gemma_9b (closest), llama_8b
    """
    tiers = {
        "~3B":   ["qwen_3b",  "gemma_2b", "llama_3b"],
        "~7–9B": ["qwen_7b",  "gemma_9b", "llama_8b"],
    }
    categories = ["capability", "behavioral", "values", "cultural", "personality"]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

    for ax, (tier_label, mks) in zip(axes, tiers.items()):
        available = [mk for mk in mks if mk in models]
        if not available:
            ax.set_title(f"{tier_label} (no data)")
            continue

        x      = np.arange(len(categories))
        n_mods = len(available)
        width  = 0.8 / n_mods

        for i, mk in enumerate(available):
            ds = models[mk].get("dim_scores", {})
            cat_scores = []
            for cat in categories:
                vals = [v["score"] for k, v in ds.items()
                        if DIM_CATEGORIES.get(k) == cat and not math.isnan(v["score"])]
                cat_scores.append(statistics.mean(vals) if vals else 0.0)

            fam = MODEL_FAMILIES.get(mk, "qwen")
            offset = (i - n_mods / 2 + 0.5) * width
            ax.bar(x + offset, cat_scores, width=width * 0.9,
                   color=PAL[fam], alpha=0.85, label=mk)

        ax.set_xticks(x)
        ax.set_xticklabels([c.capitalize() for c in categories], rotation=20, ha="right")
        ax.set_ylim(0, 1.05)
        ax.set_title(f"Matched Tier: {tier_label}")
        ax.legend(fontsize=7)
        ax.axhline(0.5, color=PAL["text_lt"], lw=0.7, ls="--", alpha=0.5)

    axes[0].set_ylabel("Mean category score")
    fig.suptitle("Cross-Family Comparison at Matched Parameter Tiers", fontsize=12)
    fig.tight_layout()
    _save(fig, 21, "matched_comparison")


# %% Chart 22 — Cultural Topology Scatter
def chart_22_cultural_scatter(models: dict):
    """
    2D scatter of cultural dimensions: individualism_collectivism (x)
    vs. inter_religious_symmetry (y), one point per model, coloured by family.
    Panel inset: all 8 cultural dim scores per model as a profile.
    """
    cultural_dims = [
        "political_framing_bias", "inter_religious_symmetry", "epistemic_avoidance",
        "geographic_knowledge", "cultural_normalization", "individualism_collectivism",
        "moral_foundation_distribution", "contested_science_calibration",
        "historical_narrative_calibration",
    ]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Left: scatter
    ax = axes[0]
    for mk, vd in models.items():
        x_score = _safe_score(vd, "individualism_collectivism")
        y_score = _safe_score(vd, "inter_religious_symmetry")
        if math.isnan(x_score) or math.isnan(y_score):
            continue
        fam = MODEL_FAMILIES.get(mk, "qwen")
        ax.scatter(x_score, y_score, s=60, color=PAL[fam], zorder=5)
        ax.annotate(
            mk.replace("_", "\n"), (x_score, y_score),
            textcoords="offset points", xytext=(5, 3), fontsize=6.5,
        )

    ax.set_xlabel("Individualism–Collectivism score")
    ax.set_ylabel("Inter-religious Symmetry score")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_title("Cultural Topology Scatter")
    ax.axhline(0.5, color=PAL["grid"], lw=0.8)
    ax.axvline(0.5, color=PAL["grid"], lw=0.8)
    fam_handles = [
        mpatches.Patch(color=PAL[f], label=f.capitalize())
        for f in ["qwen", "gemma", "llama"]
    ]
    ax.legend(handles=fam_handles, fontsize=7)

    # Right: cultural dim profile per model (line chart)
    ax2 = axes[1]
    x   = np.arange(len(cultural_dims))
    for mk, vd in models.items():
        scores = [_safe_score(vd, d) for d in cultural_dims]
        fam    = MODEL_FAMILIES.get(mk, "qwen")
        ax2.plot(x, scores, marker="o", markersize=4, lw=1.2,
                 color=PAL[fam], alpha=0.75, label=mk)

    ax2.set_xticks(x)
    ax2.set_xticklabels(
        [DIM_SHORT[d] for d in cultural_dims], rotation=40, ha="right", fontsize=7,
    )
    ax2.set_ylim(0, 1.05)
    ax2.set_ylabel("Score")
    ax2.set_title("Cultural Dimension Profiles — All Models")
    ax2.legend(fontsize=6.5, loc="lower right", ncol=2)

    fig.suptitle("Cultural Topology Analysis", fontsize=12)
    fig.tight_layout()
    _save(fig, 22, "cultural_scatter")


# %% Chart 23 — Avoidance Topology Heatmap
def chart_23_avoidance_heatmap(models: dict):
    """
    Models × epistemic_avoidance score heatmap.
    Also shows pattern-level refusal + hedge rates.
    """
    model_keys = sorted(models.keys())

    avoid_scores  = [_safe_score(models[mk], "epistemic_avoidance") for mk in model_keys]
    refusal_rates = [
        models[mk].get("pattern_summary", {}).get("epistemic_avoidance", {}).get("refusal_rate", 0)
        for mk in model_keys
    ]
    hedge_rates   = [
        models[mk].get("pattern_summary", {}).get("epistemic_avoidance", {}).get("hedge_rate", 0)
        for mk in model_keys
    ]

    fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=False)

    def _hbar(ax, values, title, color):
        y = np.arange(len(model_keys))
        ax.barh(y, values, color=color, alpha=0.85, height=0.6)
        ax.set_yticks(y)
        ax.set_yticklabels(model_keys, fontsize=8)
        ax.set_xlim(0, 1.05)
        ax.set_title(title)
        ax.axvline(0.5, color=PAL["text_lt"], lw=0.7, ls="--", alpha=0.5)
        ax.invert_yaxis()
        for i, v in enumerate(values):
            ax.text(v + 0.01, i, f"{v:.2f}", va="center", fontsize=7.5)

    _hbar(axes[0], avoid_scores,  "Avoidance Score",    PAL["cultural"])
    _hbar(axes[1], refusal_rates, "Refusal Rate",        PAL["R"])
    _hbar(axes[2], hedge_rates,   "Hedge Rate",          PAL["F"])

    fig.suptitle("Epistemic Avoidance Topology — Cross-Model", fontsize=12)
    fig.tight_layout()
    _save(fig, 23, "avoidance_heatmap")


# %% Chart 24 — Religious Symmetry Heatmap
def chart_24_religious_heatmap(models: dict):
    """
    Models × {religious_symmetry, ingroup_bias, political_framing_bias,
              moral_consistency, cultural_normalization} asymmetry heatmap.
    """
    pair_dims  = [
        "inter_religious_symmetry", "ingroup_bias",
        "political_framing_bias", "moral_consistency", "cultural_normalization",
    ]
    model_keys = sorted(models.keys())

    matrix = np.array([
        [_safe_score(models[mk], d) for d in pair_dims]
        for mk in model_keys
    ])

    fig, ax = plt.subplots(figsize=(9, max(4, len(model_keys) * 0.65 + 1.5)))
    im = ax.imshow(matrix, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)

    ax.set_xticks(np.arange(len(pair_dims)))
    ax.set_xticklabels([DIM_SHORT[d] for d in pair_dims], rotation=25, ha="right", fontsize=8.5)
    ax.set_yticks(np.arange(len(model_keys)))
    ax.set_yticklabels(model_keys, fontsize=8.5)

    for i in range(len(model_keys)):
        for j in range(len(pair_dims)):
            val = matrix[i, j]
            if not math.isnan(val):
                ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                        fontsize=8,
                        color="white" if val < 0.4 or val > 0.75 else PAL["text"])

    cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cbar.ax.tick_params(labelsize=7)
    cbar.set_label("Consistency score (0=asymmetric, 1=symmetric)", fontsize=7.5)

    ax.set_title("Symmetry / Asymmetry Heatmap — Pair Dimensions × Models", fontsize=12, pad=10)
    fig.tight_layout()
    _save(fig, 24, "religious_heatmap")


# %% Chart 25 — Big Five Radar Overlay
def chart_25_big_five_overlay(models: dict):
    """
    Overlapping radar of personality_traits + personality_consistency +
    personality_under_pressure for all models, grouped by family.
    """
    dims   = ["personality_traits", "personality_consistency", "personality_under_pressure"]
    labels = ["Big Five", "Consistency", "Under Pressure"]

    n      = len(dims)
    angles = [2 * math.pi * i / n for i in range(n)] + [0]

    fig, axes = plt.subplots(1, 3, figsize=(13, 4.5),
                             subplot_kw=dict(polar=True))
    families  = ["qwen", "gemma", "llama"]

    for ax, fam in zip(axes, families):
        ax.set_facecolor(PAL["bg"])
        fam_models = [mk for mk in sorted(models.keys()) if MODEL_FAMILIES.get(mk) == fam]

        for r in [0.2, 0.4, 0.6, 0.8, 1.0]:
            ax.plot(angles, [r] * (n + 1), color=PAL["grid"], lw=0.5)

        for mk in fam_models:
            scores = [_safe_score(models[mk], d) for d in dims]
            scores_plot = scores + [scores[0]]
            alpha  = 0.25 + 0.15 * fam_models.index(mk)
            ax.fill(angles[:-1], scores, alpha=0.12, color=PAL[fam])
            ax.plot(angles, scores_plot, lw=1.5, color=PAL[fam],
                    alpha=alpha + 0.4, label=mk)
            ax.scatter(angles[:-1], scores, s=20, color=PAL[fam],
                       alpha=alpha + 0.5, zorder=5)

        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(labels, fontsize=8)
        ax.set_yticks([])
        ax.spines["polar"].set_visible(False)
        ax.set_title(f"{fam.capitalize()} family", pad=10, fontsize=10)
        ax.legend(fontsize=6.5, loc="lower left", bbox_to_anchor=(-0.2, -0.15))

    fig.suptitle("Big Five Personality Profile — Cross-Model Overlay", fontsize=12, y=1.01)
    fig.tight_layout()
    _save(fig, 25, "big_five_overlay")


def cross_model_charts(models: dict):
    """Generate all 7 cross-model charts (19–25). Requires ≥2 models."""
    if len(models) < 2:
        print("  Need at least 2 models for cross-model charts.")
        return
    print(f"\n── Cross-model charts ({len(models)} models) ──")
    chart_19_capability_heatmap(models)
    chart_20_scaling_curves(models)
    chart_21_matched_comparison(models)
    chart_22_cultural_scatter(models)
    chart_23_avoidance_heatmap(models)
    chart_24_religious_heatmap(models)
    chart_25_big_five_overlay(models)
    print(f"  Done. Charts saved to {CHART_DIR}/")


# ─────────────────────────────────────────────────────────────────────────────
# ════════════════════════════════════════════════════════════════════════════
# SECTION C — LaTeX TABLE EXPORT
# All tables use booktabs formatting; output to paper_tables/*.tex
# ════════════════════════════════════════════════════════════════════════════
# ─────────────────────────────────────────────────────────────────────────────

# %% LaTeX helpers
def _tex_escape(s: str) -> str:
    """Escape special LaTeX characters."""
    for char, repl in [
        ("&", r"\&"), ("%", r"\%"), ("$", r"\$"), ("#", r"\#"),
        ("_", r"\_"), ("{", r"\{"), ("}", r"\}"), ("~", r"\textasciitilde{}"),
        ("^", r"\^{}"),
    ]:
        s = s.replace(char, repl)
    return s


def _save_tex(content: str, n: int, name: str):
    path = TABLE_DIR / f"table_{n:02d}_{name}.tex"
    path.write_text(content)
    print(f"  ✓ table_{n:02d}_{name}.tex")
    return path


def _booktabs_header(cols: list[str], caption: str, label: str) -> str:
    col_spec = "l" + "r" * (len(cols) - 1)
    headers  = " & ".join(_tex_escape(c) for c in cols) + r" \\"
    return textwrap.dedent(f"""
        \\begin{{table}}[htbp]
        \\centering
        \\caption{{{_tex_escape(caption)}}}
        \\label{{{label}}}
        \\small
        \\begin{{tabular}}{{{col_spec}}}
        \\toprule
        {headers}
        \\midrule
    """).lstrip()


def _booktabs_footer() -> str:
    return textwrap.dedent(r"""
        \bottomrule
        \end{tabular}
        \end{table}
    """).lstrip()


# %% Table 1 — Full Dimension Score Matrix
def table_1_score_matrix(models: dict):
    """31 dimensions × N models, all mean scores."""
    model_keys = sorted(models.keys())
    dims       = ALL_DIMS

    cols = ["Dimension", "Category"] + model_keys
    caption = "Full Dimension Score Matrix. All scores are mean across 25 test prompts (0–1)."
    label   = "tab:score_matrix"

    lines = [_booktabs_header(cols, caption, label)]

    prev_cat = None
    for dim in dims:
        cat = DIM_CATEGORIES[dim]
        if cat != prev_cat and prev_cat is not None:
            lines.append(r"\midrule" + "\n")
        prev_cat = cat

        scores = [
            f"{_safe_score(models[mk], dim):.3f}"
            if mk in models and not math.isnan(_safe_score(models[mk], dim))
            else "--"
            for mk in model_keys
        ]
        row = " & ".join(
            [_tex_escape(DIM_SHORT.get(dim, dim)), _tex_escape(cat)] + scores
        )
        lines.append(row + r" \\" + "\n")

    lines.append(_booktabs_footer())
    _save_tex("".join(lines), 1, "score_matrix")


# %% Table 2 — Cliff Analysis
def table_2_cliff_analysis(models: dict):
    """
    Dimension–model pairs where score drops ≥0.25 versus the next
    larger model in the same family (cliff = parameter-count regression).
    """
    families    = ["qwen", "gemma", "llama"]
    fam_order   = {
        "qwen":  ["qwen_0.5b", "qwen_1.5b", "qwen_3b", "qwen_7b"],
        "gemma": ["gemma_2b", "gemma_9b"],
        "llama": ["llama_3b", "llama_8b"],
    }

    cols    = ["Dimension", "Family", "Smaller model", "Larger model",
               "Score (small)", "Score (large)", "Drop"]
    caption = ("Cliff analysis: dimension--model pairs where a larger model "
               "scores $\\geq 0.25$ lower than the next smaller model in the same family.")
    label   = "tab:cliff_analysis"

    lines = [_booktabs_header(cols, caption, label)]
    found = False

    for dim in ALL_DIMS:
        for fam in families:
            mks = [mk for mk in fam_order.get(fam, []) if mk in models]
            for i in range(len(mks) - 1):
                s_small = _safe_score(models[mks[i]],     dim)
                s_large = _safe_score(models[mks[i + 1]], dim)
                if math.isnan(s_small) or math.isnan(s_large):
                    continue
                drop = s_large - s_small   # negative = regression
                if drop <= -0.25:
                    row = " & ".join([
                        _tex_escape(DIM_SHORT.get(dim, dim)),
                        _tex_escape(fam),
                        _tex_escape(mks[i]),
                        _tex_escape(mks[i + 1]),
                        f"{s_small:.3f}",
                        f"{s_large:.3f}",
                        f"{drop:.3f}",
                    ])
                    lines.append(row + r" \\" + "\n")
                    found = True

    if not found:
        lines.append(r"(No cliffs $\geq 0.25$ detected) \\" + "\n")

    lines.append(_booktabs_footer())
    _save_tex("".join(lines), 2, "cliff_analysis")


# %% Table 3 — Asymmetry Findings
def table_3_asymmetries(models: dict):
    """All detected asymmetries across pair-scored dimensions."""
    cols    = ["Model", "Dimension", "Pair ID",
               "Consistency", "Description"]
    caption = ("Detected asymmetries in symmetric-pair dimensions. "
               "Consistency score: 1.0 = perfectly symmetric. "
               "Descriptions are verbatim frontier model observations.")
    label   = "tab:asymmetries"

    lines = [_booktabs_header(cols, caption, label)]
    found = False

    for mk in sorted(models.keys()):
        for asym in models[mk].get("asymmetries", []):
            desc = asym.get("description", "")[:120]   # truncate for table width
            if not desc:
                continue
            row = " & ".join([
                _tex_escape(mk),
                _tex_escape(DIM_SHORT.get(asym.get("dimension", ""), asym.get("dimension", ""))),
                _tex_escape(str(asym.get("pair_id", ""))),
                f"{asym.get('consistency_score', float('nan')):.3f}"
                    if asym.get("consistency_score") is not None else "--",
                _tex_escape(desc),
            ])
            lines.append(row + r" \\" + "\n")
            found = True

    if not found:
        lines.append(r"(No asymmetries detected) \\" + "\n")

    lines.append(_booktabs_footer())
    _save_tex("".join(lines), 3, "asymmetry_findings")


# %% Table 4 — Hypothesis Evidence Summary
def table_4_hypotheses(models: dict):
    """Per-hypothesis evidence extracted from each model's synthesis."""
    hypotheses = ["H1", "H2", "H3", "H4", "H5", "H6"]
    hyp_labels = {
        "H1": "Capability $\\propto$ params (within family)",
        "H2": "Cultural topology $\\sim$ corpus (not params)",
        "H3": "Moral calibration improves with scale",
        "H4": "Personality stability improves with scale",
        "H5": "Avoidance topology differs by family",
        "H6": "Religious asymmetry direction differs by family",
    }

    cols    = ["Hypothesis", "Model", "Evidence (excerpt)"]
    caption = ("Evidence for each research hypothesis extracted from behavioral synthesis. "
               "Evidence is the frontier model's verbatim assessment.")
    label   = "tab:hypothesis_evidence"

    lines = [_booktabs_header(cols, caption, label)]

    for hyp in hypotheses:
        first = True
        for mk in sorted(models.keys()):
            evidence = (
                models[mk]
                .get("synthesis", {})
                .get("hypothesis_evidence", {})
                .get(hyp, "")
            )
            if not evidence:
                continue
            excerpt = evidence[:150]
            hyp_cell = _tex_escape(f"{hyp}: {hyp_labels[hyp]}") if first else ""
            row = " & ".join([
                hyp_cell,
                _tex_escape(mk),
                _tex_escape(excerpt),
            ])
            lines.append(row + r" \\" + "\n")
            first = False
        lines.append(r"\midrule" + "\n")

    lines.append(_booktabs_footer())
    _save_tex("".join(lines), 4, "hypothesis_evidence")


# %% Table 5 — Avoidance Topology
def table_5_avoidance_topology(models: dict):
    """
    Models × epistemic avoidance score + refusal/hedge rates.
    Per-topic category breakdown is sparse in current data; we report
    aggregate scores and signal rates.
    """
    model_keys = sorted(models.keys())
    cols       = ["Model", "Family", "Params (B)",
                  "Avoid. score", "Refusal rate", "Hedge rate", "Asymmetries"]
    caption    = ("Epistemic avoidance topology. "
                  "Scores are frontier-model assessments (0--1). "
                  "Refusal and hedge rates are structural surface signals.")
    label      = "tab:avoidance_topology"

    lines = [_booktabs_header(cols, caption, label)]

    for mk in model_keys:
        vd  = models[mk]
        ps  = vd.get("pattern_summary", {}).get("epistemic_avoidance", {})
        n_asym = sum(
            1 for a in vd.get("asymmetries", [])
            if a.get("dimension") == "epistemic_avoidance"
        )
        row = " & ".join([
            _tex_escape(mk),
            _tex_escape(MODEL_FAMILIES.get(mk, "?")),
            f"{MODEL_PARAMS.get(mk, '?')}",
            f"{_safe_score(vd, 'epistemic_avoidance'):.3f}",
            f"{ps.get('refusal_rate', 0):.3f}",
            f"{ps.get('hedge_rate',   0):.3f}",
            str(n_asym),
        ])
        lines.append(row + r" \\" + "\n")

    lines.append(_booktabs_footer())
    _save_tex("".join(lines), 5, "avoidance_topology")


def export_all_tables(models: dict):
    """Export all 5 LaTeX tables."""
    print(f"\n── LaTeX table export ({len(models)} models) ──")
    table_1_score_matrix(models)
    table_2_cliff_analysis(models)
    table_3_asymmetries(models)
    table_4_hypotheses(models)
    table_5_avoidance_topology(models)
    print(f"  Done. Tables saved to {TABLE_DIR}/")


# ─────────────────────────────────────────────────────────────────────────────
# %% Master Runners
# ─────────────────────────────────────────────────────────────────────────────

def run_single_model(viz_data_path: str | Path):
    """Load one viz_data_v2.json and generate charts 12–18."""
    viz_data = load_viz_data(viz_data_path)
    single_model_charts(viz_data)


def run_cross_model(
    registry_path: str | Path = "profile_runs/registry_results_index.json",
    tables: bool = True,
):
    """Load all completed models and generate charts 19–25 + tables 1–5."""
    models = load_registry(registry_path)
    if not models:
        print("No completed models found in registry.")
        return
    cross_model_charts(models)
    if tables:
        export_all_tables(models)


def run_all(
    registry_path: str | Path = "profile_runs/registry_results_index.json",
    single_model_key: str | None = None,
):
    """
    Full visualization run:
      - If single_model_key is given, also generates charts 12–18 for that model.
      - Always generates cross-model charts 19–25 and tables 1–5.
    """
    models = load_registry(registry_path)
    if not models:
        print("No completed models found in registry.")
        return

    if single_model_key:
        if single_model_key in models:
            single_model_charts(models[single_model_key])
        else:
            print(f"  {single_model_key} not found in loaded models.")

    if len(models) >= 2:
        cross_model_charts(models)
        export_all_tables(models)
    else:
        print("  Need ≥2 completed models for cross-model charts.")


# ─────────────────────────────────────────────────────────────────────────────
# %% Usage Examples (uncomment to run)
# ─────────────────────────────────────────────────────────────────────────────
# Single model only:
run_single_model("profile_qwen_0.5b_20260319_113802/viz/viz_data_v2.json")
#
# Cross-model only (charts + tables):
#   run_cross_model()
#
# Everything, with qwen_0.5b as the exemplar single-model:
#   run_all(single_model_key="qwen_0.5b")


── Single-model charts: qwen_0.5b ──
  ✓ chart_12_radar_31dim.pdf
  ✓ chart_13_values_panel.pdf
  ✓ chart_14_cultural_bar.pdf
  ✓ chart_15_epistemic_avoidance.pdf
  ✓ chart_16_symmetry_asymmetry.pdf
  ✓ chart_17_big_five_radar.pdf
  ✓ chart_18_personality_pressure.pdf

── Medium table images: qwen_0.5b ──
  ✓ table_capability_qwen_0.5b.png
  ✓ table_behavioral_qwen_0.5b.png
  ✓ table_values_qwen_0.5b.png
  ✓ table_cultural_qwen_0.5b.png
  ✓ table_personality_qwen_0.5b.png
  ✓ table_symmetry_qwen_0.5b.png
  ✓ table_all_dims_qwen_0.5b.png
  ✓ table_methodology_qwen_0.5b.png
  Done. Table images saved to charts/
  Done. Charts saved to charts/
